# Model Benchmark (Grid Search) — from prediction exports
Put this notebook into your project root (e.g., `D:\\progr`) and **Run All**.

✅ This notebook **does not require** `data/dataset.parquet`.

It will:
- scan `outputs/predictions/*_w*_fisher_z_predictions.csv`
- build lag features from `y_true`
- run **GridSearchCV** (time-series splits) for Ridge / RandomForest / XGBoost(GPU)
- compare vs baselines present in the same CSV (e.g., `DCC_GARCH`, `AR1`, `Naive_Last`)
- export results to:
  - `outputs/results/gridsearch_benchmarks.csv`
  - `outputs/tables_notebook/gridsearch_benchmarks.tex`


In [1]:

from __future__ import annotations

import re
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# XGBoost is optional (GPU)
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    XGB_IMPORT_ERROR = repr(e)

ROOT = Path(".").resolve()
PRED_DIR = ROOT / "outputs" / "predictions"
OUT_RESULTS = ROOT / "outputs" / "results"
OUT_TABLES = ROOT / "outputs" / "tables_notebook"

OUT_RESULTS.mkdir(parents=True, exist_ok=True)
OUT_TABLES.mkdir(parents=True, exist_ok=True)

print("Notebook ROOT:", ROOT)
print("Pred dir:", PRED_DIR)
print("Exists pred dir?", PRED_DIR.exists())
print("XGBoost available?", HAS_XGB, ("" if HAS_XGB else f" | import error: {XGB_IMPORT_ERROR}"))


Notebook ROOT: D:\progr
Pred dir: D:\progr\outputs\predictions
Exists pred dir? True
XGBoost available? True 


In [2]:

# ---------- helpers ----------
def parse_dep_and_window(filename: str) -> tuple[str, int | None]:
    m = re.search(r"^(.*)_w(\d+)_", filename)
    if not m:
        return filename.replace("_predictions.csv",""), None
    return m.group(1), int(m.group(2))

def build_lag_features(df: pd.DataFrame, target_col: str = "y_true", lags: int = 30):
    data = df.copy()
    data[target_col] = pd.to_numeric(data[target_col], errors="coerce")
    for k in range(1, lags+1):
        data[f"lag_{k}"] = data[target_col].shift(k)
    data = data.dropna(subset=[target_col] + [f"lag_{k}" for k in range(1, lags+1)]).reset_index(drop=True)
    X = data[[f"lag_{k}" for k in range(1, lags+1)]].to_numpy(dtype=float)
    y = data[target_col].to_numpy(dtype=float)
    return X, y, data

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate(y_true, y_pred) -> dict:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "R2": float(r2_score(y_true, y_pred)),
        "n": int(len(y_true)),
    }


In [3]:

# ---------- locate prediction files ----------
files = sorted(PRED_DIR.glob("*_w*_fisher_z_predictions.csv"))
print("Found prediction files:", len(files))
for f in files[:12]:
    print(" -", f.name)

assert len(files) > 0, f"No files found in {PRED_DIR}. Run your pipeline first (python main.py)."

sample = pd.read_csv(files[0])
print("\nSample file:", files[0].name)
print("Columns:", list(sample.columns))
assert "y_true" in sample.columns, "Expected column 'y_true' in prediction exports."


Found prediction files: 24
 - corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w60_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w90_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w14_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w30_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w60_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w90_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w14_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w30_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w60_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w90_fisher_z_predictions.csv

Sample file: corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv
Columns: ['Date', 'y_true', 'Naive_Last', 'AR1', 'ElasticNet', 'Ridge', 'RF', 'XGB_GPU', 'DCC_GARCH']


## Settings

In [4]:

LAGS = 30
TEST_FRAC = 0.20
N_SPLITS = 5

MAX_FILES = None  # e.g. 5 for quick test

BASELINE_COLS = ["Naive_Last", "AR1", "DCC_GARCH"]  # evaluated if present


## Parameter grids

In [5]:

ridge_param_grid = {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 300.0], "fit_intercept": [True, False]}

rf_param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 3, 5],
    "random_state": [42],
}

xgb_param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.07, 0.1],
    "n_estimators": [300, 600],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0],
}


## Grid search loop (time-series CV) + baseline comparison

In [6]:

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

rows = []
processed = 0

for path in files:
    if MAX_FILES is not None and processed >= MAX_FILES:
        break

    df = pd.read_csv(path)

    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.sort_values("Date").reset_index(drop=True)

    dep, w = parse_dep_and_window(path.name)

    X, y, df_feat = build_lag_features(df, target_col="y_true", lags=LAGS)
    if len(y) < max(80, (N_SPLITS + 1) * 10):
        print(f"SKIP {path.name}: too few rows after lagging ({len(y)})")
        continue

    split = int(len(y) * (1.0 - TEST_FRAC))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # ----- tuned Ridge -----
    gs_ridge = GridSearchCV(Ridge(), ridge_param_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=-1)
    gs_ridge.fit(X_train, y_train)
    best_ridge = gs_ridge.best_estimator_
    met = evaluate(y_test, best_ridge.predict(X_test))
    rows.append({"dependency": dep, "window": w, "file": path.name, "model": "Ridge_tuned",
                 "best_params": str(gs_ridge.best_params_), **met})

    # ----- tuned RF -----
    gs_rf = GridSearchCV(RandomForestRegressor(), rf_param_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=-1)
    gs_rf.fit(X_train, y_train)
    best_rf = gs_rf.best_estimator_
    met = evaluate(y_test, best_rf.predict(X_test))
    rows.append({"dependency": dep, "window": w, "file": path.name, "model": "RF_tuned",
                 "best_params": str(gs_rf.best_params_), **met})

    # ----- tuned XGB GPU (optional) -----
    if HAS_XGB:
        xgb_est = xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", device="cuda", random_state=42)
        gs_xgb = GridSearchCV(xgb_est, xgb_param_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=1)
        gs_xgb.fit(X_train, y_train)
        best_xgb = gs_xgb.best_estimator_
        met = evaluate(y_test, best_xgb.predict(X_test))
        rows.append({"dependency": dep, "window": w, "file": path.name, "model": "XGB_GPU_tuned",
                     "best_params": str(gs_xgb.best_params_), **met})
    else:
        rows.append({"dependency": dep, "window": w, "file": path.name, "model": "XGB_GPU_tuned",
                     "best_params": f"SKIPPED: {XGB_IMPORT_ERROR}", "MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "n": int(len(y_test))})

    # ----- baselines from file (if present) -----
    start_idx = LAGS  # alignment after shifting
    for bcol in [c for c in BASELINE_COLS if c in df.columns]:
        b = pd.to_numeric(df.loc[start_idx:, bcol], errors="coerce").to_numpy()
        b = b[:len(y)]
        mask = np.isfinite(b)
        y_al = y[mask]
        b_al = b[mask]
        if len(y_al) < 50:
            continue
        split_b = int(len(y_al) * (1.0 - TEST_FRAC))
        yb_test = y_al[split_b:]
        bb_test = b_al[split_b:]
        if len(yb_test) < 10:
            continue
        metb = evaluate(yb_test, bb_test)
        rows.append({"dependency": dep, "window": w, "file": path.name, "model": f"baseline_{bcol}",
                     "best_params": "", **metb})

    processed += 1
    print(f"Done: {dep} | w={w} | test_n={len(y_test)}")

results = pd.DataFrame(rows)
print("Total rows:", len(results))
display(results.head(20))


D:\progr\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [14:28:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Done: corr_BTC-USD_^GSPC | w=14 | test_n=405
Done: corr_BTC-USD_^GSPC | w=30 | test_n=402
Done: corr_BTC-USD_^GSPC | w=60 | test_n=396
Done: corr_BTC-USD_^GSPC | w=90 | test_n=390
Done: corr_BTC-USD_^IXIC | w=14 | test_n=405
Done: corr_BTC-USD_^IXIC | w=30 | test_n=402
Done: corr_BTC-USD_^IXIC | w=60 | test_n=396
Done: corr_BTC-USD_^IXIC | w=90 | test_n=390
Done: corr_BTC-USD_ETH-USD | w=14 | test_n=405
Done: corr_BTC-USD_ETH-USD | w=30 | test_n=402
Done: corr_BTC-USD_ETH-USD | w=60 | test_n=396
Done: corr_BTC-USD_ETH-USD | w=90 | test_n=390
Done: corr_BTC-USD_GLD | w=14 | test_n=405
Done: corr_BTC-USD_GLD | w=30 | test_n=402
Done: corr_BTC-USD_GLD | w=60 | test_n=396
Done: corr_BTC-USD_GLD | w=90 | test_n=390
Done: corr_BTC-USD_SLV | w=14 | test_n=405
Done: corr_BTC-USD_SLV | w=30 | test_n=402
Done: corr_BTC-USD_SLV | w=60 | test_n=396
Done: corr_BTC-USD_SLV | w=90 | test_n=390
Done: corr_BTC-USD_UUP | w=14 | test_n=405
Done: corr_BTC-USD_UUP | w=30 | test_n=402
Done: corr_BTC-USD_UUP

,dependency,window,file,model,best_params,MAE,RMSE,R2,n
0,corr_BTC-USD_^GSPC,14,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv,Ridge_tuned,"{'alpha': 0.1, 'fit_intercept': True}",0.092097,0.133495,0.845976,405
1,corr_BTC-USD_^GSPC,14,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv,RF_tuned,"{'max_depth': 5, 'min_samples_leaf': 5, 'n_est...",0.100242,0.146174,0.815328,405
2,corr_BTC-USD_^GSPC,14,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv,XGB_GPU_tuned,"{'colsample_bytree': 1.0, 'learning_rate': 0.0...",0.100898,0.144557,0.819392,405
3,corr_BTC-USD_^GSPC,14,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv,baseline_Naive_Last,,0.094194,0.144216,0.826374,251
4,corr_BTC-USD_^GSPC,14,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv,baseline_AR1,,0.094551,0.142127,0.831367,251
5,corr_BTC-USD_^GSPC,14,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv,baseline_DCC_GARCH,,0.247764,0.313846,0.148679,405
6,corr_BTC-USD_^GSPC,30,corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv,Ridge_tuned,"{'alpha': 0.1, 'fit_intercept': True}",0.041359,0.066909,0.902477,402
7,corr_BTC-USD_^GSPC,30,corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv,RF_tuned,"{'max_depth': 5, 'min_samples_leaf': 5, 'n_est...",0.042103,0.067067,0.902018,402
8,corr_BTC-USD_^GSPC,30,corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv,XGB_GPU_tuned,"{'colsample_bytree': 1.0, 'learning_rate': 0.0...",0.042118,0.067478,0.900812,402
9,corr_BTC-USD_^GSPC,30,corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv,baseline_Naive_Last,,0.040877,0.068552,0.903092,248


## Best model per (dependency, window) by RMSE

In [7]:

res_ok = results.dropna(subset=["RMSE"]).copy()
best = (res_ok.sort_values(["dependency", "window", "RMSE"])
        .groupby(["dependency", "window"], as_index=False)
        .first())

display(best)

out_csv = OUT_RESULTS / "gridsearch_benchmarks.csv"
best.to_csv(out_csv, index=False)
print("Saved:", out_csv)


,dependency,window,file,model,best_params,MAE,RMSE,R2,n
0,corr_BTC-USD_ETH-USD,14,corr_BTC-USD_ETH-USD_w14_fisher_z_predictions.csv,Ridge_tuned,"{'alpha': 1.0, 'fit_intercept': True}",0.098027,0.141863,0.875707,405
1,corr_BTC-USD_ETH-USD,30,corr_BTC-USD_ETH-USD_w30_fisher_z_predictions.csv,baseline_AR1,,0.041490,0.061882,0.964490,248
2,corr_BTC-USD_ETH-USD,60,corr_BTC-USD_ETH-USD_w60_fisher_z_predictions.csv,baseline_Naive_Last,,0.019242,0.030228,0.982785,242
3,corr_BTC-USD_ETH-USD,90,corr_BTC-USD_ETH-USD_w90_fisher_z_predictions.csv,baseline_AR1,,0.012663,0.020148,0.985480,236
4,corr_BTC-USD_GLD,14,corr_BTC-USD_GLD_w14_fisher_z_predictions.csv,Ridge_tuned,"{'alpha': 0.1, 'fit_intercept': True}",0.084592,0.127751,0.850464,405
5,corr_BTC-USD_GLD,30,corr_BTC-USD_GLD_w30_fisher_z_predictions.csv,baseline_AR1,,0.037148,0.057305,0.881068,248
6,corr_BTC-USD_GLD,60,corr_BTC-USD_GLD_w60_fisher_z_predictions.csv,baseline_AR1,,0.018781,0.031178,0.921470,242
7,corr_BTC-USD_GLD,90,corr_BTC-USD_GLD_w90_fisher_z_predictions.csv,Ridge_tuned,"{'alpha': 0.01, 'fit_intercept': True}",0.013255,0.023316,0.934094,390
8,corr_BTC-USD_SLV,14,corr_BTC-USD_SLV_w14_fisher_z_predictions.csv,Ridge_tuned,"{'alpha': 0.1, 'fit_intercept': True}",0.079052,0.113205,0.866801,405
9,corr_BTC-USD_SLV,30,corr_BTC-USD_SLV_w30_fisher_z_predictions.csv,baseline_AR1,,0.035996,0.054292,0.885187,248


Saved: D:\progr\outputs\results\gridsearch_benchmarks.csv


## Export LaTeX table (safe for underscores)

In [8]:

def latex_escape(s: str) -> str:
    s = str(s)
    s = s.replace("\\", "\\textbackslash ")
    s = s.replace("_", "\\_")
    s = s.replace("%", "\\%")
    s = s.replace("&", "\\&")
    s = s.replace("#", "\\#")
    return s

tab = best.copy()
tab["dependency"] = tab["dependency"].map(latex_escape)
tab["model"] = tab["model"].map(latex_escape)

def shorten_params(x, maxlen=90):
    x = str(x)
    return x if len(x) <= maxlen else x[:maxlen] + "..."

tab["best_params"] = tab["best_params"].map(lambda x: latex_escape(shorten_params(x)))

tab = tab[["dependency", "window", "model", "MAE", "RMSE", "R2", "n", "best_params"]].copy()

tex_path = OUT_TABLES / "gridsearch_benchmarks.tex"
latex = tab.to_latex(index=False, float_format="%.6f", escape=False, longtable=True,
                     caption="Best-performing models per dependency and rolling window (selected by minimum RMSE on the test split). Tuned models use time-series cross-validation.",
                     label="tab:gridsearch_best")
tex_path.write_text(latex, encoding="utf-8")
print("Saved LaTeX table:", tex_path)


Saved LaTeX table: D:\progr\outputs\tables_notebook\gridsearch_benchmarks.tex
